# IsoFormer sequential attention on the BLF codon grid

How does one fusion method let the result of reading RNA change how it reads protein? This Stage 3 reference follows the released IsoFormer update order and places it between the BioLangFusion (BLF) codon grid and prediction head. The examples use invented feature arrays to check the computation before any benchmark comparison.

## Setup

Use the repository's Python environment and start the kernel from the repository root or this notebook's folder. The examples initialize small models on the CPU and use a fixed seed to make the illustrated calculations repeatable. [Notebook dependencies](../../README.md#notebook-dependencies)

In [ ]:
from pathlib import Path
import sys

import numpy as np  # Import before PyTorch on Windows to avoid an OpenMP conflict.
import torch

# Find Code/shared_code without changing the kernel's working directory.
working_dir = Path.cwd().resolve()
for folder in (working_dir, *working_dir.parents):
    if (folder / "shared_code" / "fusion").is_dir():
        code_dir = folder
        break
    if (folder / "Code" / "shared_code" / "fusion").is_dir():
        code_dir = folder / "Code"
        break
else:
    raise FileNotFoundError("Open the kernel from inside the repository.")

if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

from shared_code.fusion.codon_grid import CodonGrid
from shared_code.fusion.heads import (
    N_HEADS, SHARED_WIDTH, CrossModalFusion, IsoFormerSequentialFusion,
)
from shared_code.fusion.textcnn import DEFAULT_CHANNELS, TextCNN

EXAMPLE_SEED = 0
rng = np.random.default_rng(EXAMPLE_SEED)
torch.manual_seed(EXAMPLE_SEED)

---

## Why update DNA twice?

The released IsoFormer first uses DNA tokens as queries over the RNA tokens. Each DNA token gets a weighted summary of RNA values, which is projected and added back to the DNA token. The protein block then uses these **RNA-updated DNA tokens** as its queries. RNA can therefore change how protein tokens are weighted, even when the protein features stay fixed. The fused stream stays on the DNA positions throughout. [Released IsoFormer forward pass](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py) · [Local residual attention block](../shared_code/fusion/isoformer_reference.py)

The example uses two invented rows with two and three real codons. It applies the updates explicitly and checks that they reproduce the shared head. Subtracting the protein block's input from its output isolates the update it added. Compare that update for the original and RNA-updated DNA queries while keeping the protein context and block weights fixed. [Sequential fusion adapter](../shared_code/fusion/heads.py)

In [ ]:
toy_widths = {"dna": 2, "rna": 3, "prot": 4}
toy_mask = torch.tensor([[True, True, False], [True, True, True]])
toy_streams = {
    modality: torch.from_numpy(rng.standard_normal((2, 3, width), dtype=np.float32))
    for modality, width in toy_widths.items()
}
toy_fusion = IsoFormerSequentialFusion(toy_widths, out_width=4, n_heads=1).eval()

with torch.inference_mode():
    projected = {
        modality: toy_fusion.project[modality](tokens)
        for modality, tokens in toy_streams.items()
    }
    dna = projected["dna"]
    dna_after_rna = toy_fusion.read_rna(dna, projected["rna"], context_mask=toy_mask)
    dna_after_both = toy_fusion.read_protein(
        dna_after_rna, projected["prot"], context_mask=toy_mask,
    )
    dna_after_protein_alone = toy_fusion.read_protein(
        dna, projected["prot"], context_mask=toy_mask,
    )

    fused, _ = toy_fusion(toy_streams, toy_mask)
    torch.testing.assert_close(fused[toy_mask], dna_after_both[toy_mask])

    protein_update_after_rna = dna_after_both - dna_after_rna
    protein_update_from_dna = dna_after_protein_alone - dna
    update_delta = protein_update_after_rna - protein_update_from_dna
    largest_change = update_delta[toy_mask].abs().max().item()

assert largest_change > 1e-5
print(f"Largest change in a real feature of the protein update: {largest_change:.4f}")

## What happens to padded positions?

The codon grid pads shorter rows to the longest row in the batch. Its Boolean mask marks which codons are real. The adaptation applies that mask to RNA and protein when each serves as attention context, then clears padded DNA output before the prediction head. The released fusion call passes a mask to the RNA block. Masking the protein context at fusion is an explicit choice for these padded BLF batches. [Released IsoFormer forward pass](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py)

Replace the padded features in each auxiliary stream in turn, then check the shorter row on its own with no padding. Its real DNA features should match across these checks, and the padded DNA output should be zero.

In [ ]:
with torch.inference_mode():
    for modality in ("rna", "prot"):
        changed_streams = dict(toy_streams)
        changed_streams[modality] = toy_streams[modality].masked_fill(
            ~toy_mask.unsqueeze(-1), 10.0,
        )
        fused_with_changed_padding, _ = toy_fusion(changed_streams, toy_mask)
        torch.testing.assert_close(fused_with_changed_padding[toy_mask], fused[toy_mask])

    short_streams = {name: tokens[:1, :2] for name, tokens in toy_streams.items()}
    fused_short, _ = toy_fusion(short_streams, toy_mask[:1, :2])
    torch.testing.assert_close(fused_short, fused[:1, :2])

torch.testing.assert_close(fused[~toy_mask], torch.zeros_like(fused[~toy_mask]))
print("The shorter row agrees with and without padding.")

---

## How the head fits the BLF pipeline

The released IsoFormer pools its fused DNA features over a selected window before predicting expression. Our benchmark adaptation uses the BLF construction on either side of the two attention updates. For the original encoder combination, the grid uses Equation 1 to upsample DNA six-mer features, averages RNA nucleotide features in groups of three, and places protein features at codon resolution. Each stream then gets a linear projection with a tanh activation to width 600. The adapter uses four attention heads and sends the final DNA stream into the same 100-channel TextCNN used by the BLF baselines. [Released IsoFormer forward pass](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py) · [BLF Equation 1](https://arxiv.org/html/2506.08936v1#S2.SS1) · [Local TextCNN](../shared_code/fusion/textcnn.py)

Here the feature widths match the original BLF encoders, while the values are invented. Two rows with eight and six codons should yield a fused tensor with dimensions batch × codons × features, or 2 × 8 × 600. The three-output readout illustrates the low, medium and high E. coli protein-production classes. [BLF input widths](blf_build_check.ipynb) · [E. coli target guide](../../docs/methods/data-and-targets.md)

In [ ]:
widths = {"dna": 512, "rna": 640, "prot": 320}
codon_lengths = torch.tensor([8, 6])
dna_token_counts = torch.tensor([4, 3])
dna_tokens = torch.randn(2, 4, widths["dna"])
rna_tokens = torch.randn(2, 24, widths["rna"])
protein_tokens = torch.randn(2, 8, widths["prot"])

codon_grid = CodonGrid(widths["dna"], tconv="equation").eval()
sequential_fusion = IsoFormerSequentialFusion(
    widths, out_width=SHARED_WIDTH, n_heads=N_HEADS,
).eval()
prediction_head = TextCNN(SHARED_WIDTH, n_outputs=3, channels=DEFAULT_CHANNELS).eval()

with torch.inference_mode():
    aligned_streams, codon_mask = codon_grid(
        dna_tokens, rna_tokens, protein_tokens, codon_lengths, dna_token_counts,
    )
    fused_dna, _ = sequential_fusion(aligned_streams, codon_mask)
    class_scores = prediction_head(fused_dna, codon_mask)

torch.testing.assert_close(codon_mask.sum(dim=1), codon_lengths)
assert fused_dna.shape == (2, 8, SHARED_WIDTH)
assert class_scores.shape == (2, 3)
assert torch.isfinite(class_scores).all()
print(f"Fused DNA: {tuple(fused_dna.shape)} | Class scores: {tuple(class_scores.shape)}")

joint_attention = CrossModalFusion(widths, out_width=SHARED_WIDTH, n_heads=N_HEADS)
print("Fusion parameters for the original BLF embedding widths")
print(f"  Sequential attention  {sum(p.numel() for p in sequential_fusion.parameters()):,}")
print(f"  BLF joint attention   {sum(p.numel() for p in joint_attention.parameters()):,}")

## What this reference supports

The synthetic checks cover the update order, padding and grid-to-predictor shapes. The parameter counts cover both complete fusion modules, including their projections and attention. A benchmark that fixes the inputs and TextCNN would still compare fusion heads with different capacities. A comparison with matched parallel or reversed updates could test what the order itself contributes. [Stage 4 encoder comparisons](../stage4/encoder_swaps.ipynb) · [IsoFormer source notes](../../docs/sources.md#isoformer-dataset-and-model-cards)

The earlier frozen GTEx pilot updates multiple streams and predicts transcript expression. Its results belong to that separate experiment. [Frozen GTEx pilot](gtex_frozen_fusion_reference.ipynb)